# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Each Croissant Dataset is structured with one or more record sets (tables), each with its own unique `@id`.

We'll enumerate the record sets and their field (column) `@id`s.

In [ ]:
record_sets_info = []

for rset in dataset.record_sets:
    print(f"Record Set name: {rset.name}")
    print(f"  ID: {rset.id}")
    print(f"  Description: {rset.description}")
    print("  Fields:")
    for fld in rset.fields:
        print(f"    - {fld.name} (id: {fld.id}, type: {fld.data_type})")
    record_sets_info.append({'id': rset.id, 'name': rset.name, 'fields': [f.id for f in rset.fields]})
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets into DataFrames using their @id
dataframes = {}
for rset in dataset.record_sets:
    print(f"Loading records for Record Set: {rset.name} (id: {rset.id})")
    df = pd.DataFrame(dataset.records(record_set=rset.id))
    print(f"  {df.shape[0]} records loaded. Columns: {df.columns.tolist()}")
    dataframes[rset.id] = df
    print()
# Choose the main table (first one, as commonly mass-spec protein abundance is main)
main_record_set_id = dataset.record_sets[0].id if dataset.record_sets else None
if main_record_set_id:
    print(f"Main record set chosen: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's choose a numeric field, filter, normalize, and group as example exploratory steps.

In [ ]:
# --- Configuration: define field @ids ---
# Identify a numeric field and a group field based on overview above
# Example: 'cr:field_abundance' (abundance), 'cr:field_mw' (Molecular Weight), 'cr:field_sample' (sample label)

rset = dataset.record_sets[0]
# Pick the first FLOAT/NUMBER-type field for numeric analysis
numeric_field_id = None
for f in rset.fields:
    if f.data_type is not None and f.data_type.lower() in ('float', 'number', 'integer'):
        numeric_field_id = f.id
        break
if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    print(f"Using numeric field: {numeric_field_id}")

group_field_id = None
for f in rset.fields:
    # Look for something like sample/condition/type etc:
    if f.data_type and f.data_type.lower() in ('text', 'string', 'categorical') and 'sample' in f.name.lower():
        group_field_id = f.id
        break

if group_field_id is None and len(rset.fields) > 1:
    group_field_id = rset.fields[1].id

df = dataframes[rset.id]

# Ensure field is present in the DataFrame
if numeric_field_id in df.columns:
    # Convert field to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as example threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean','count']).sort_values('mean', ascending=False)
        print(f"Grouped data by {group_field_id} (mean/count):")
        print(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in data columns:", df.columns.tolist())

## 5. Visualization

Visualize the distribution of the chosen numeric field, and a boxplot grouped by the grouping field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We demonstrated loading, overviewing, and extracting the main protein records from the Croissant FAIR^2 dataset using `mlcroissant`.
- We summarized and visualized one of the quantitative fields, grouping by a possible categorical field for deeper insights.
- You can repeat similar EDA steps for other fields or record sets, or save the processed DataFrame for downstream analysis.

**Next Steps:**
- Explore additional record sets from the dataset, if present.
- Integrate these steps into your ML workflow for protein quantification or biomarker discovery.